# Random Forest và XGBoost (cửa sổ 28 ngày)
Dự đoán `label_withdrawn` = 1 cho sinh viên còn học ở đầu ngày 28, từ thông tin ngày 0–27. Fit trên train, đánh giá baseline trên dev ở ngưỡng 0,5. Không dùng test ở giai đoạn này.

Hai mô hình dùng cùng 33 cột gốc và nhánh `tree` của preprocessor chung (49 feature sau biến đổi). Mã huấn luyện dùng lại nằm trong `src/models/baseline_rf_xgb_real.py`.

**Dữ liệu:** 19.264 lượt đăng ký thuộc tập train; 4.131 lượt thuộc dev. `feature_28d.csv`, `split.csv` và `preprocessor.joblib` nằm trong `dataset/`. Đường dẫn dữ liệu giữ nguyên như baseline hiện tại.

In [1]:
import sys
import warnings
from pathlib import Path
from sklearn.exceptions import InconsistentVersionWarning

# Artifact được lưu bằng scikit-learn 1.9.0; chỉ ẩn cảnh báo phiên bản khi đọc.
warnings.filterwarnings('ignore', category=InconsistentVersionWarning)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.models.baseline_rf_xgb_real import run
DATASET = ROOT / 'dataset'
RESULTS = ROOT / 'results'
assert DATASET.is_dir(), f'Không tìm thấy thư mục {DATASET}'

## Baseline trên train và dev
RF và XGBoost dùng cấu hình cố định 100 cây. Hai mô hình nhận cùng dữ liệu sau biến đổi; đây là mốc ban đầu trước khi tinh chỉnh tham số và ngưỡng.

In [2]:
metrics, importance, raw_features = run(DATASET, RESULTS)
print(f'{len(raw_features)} cột gốc → {metrics.n_features.iloc[0]} feature cho mỗi mô hình')
metrics

33 cột gốc → 49 feature cho mỗi mô hình


,model,split,n_train,n_dev,n_features,threshold,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp
0,random_forest,dev,19264,4131,49,0.5,0.422139,0.299601,0.350467,0.694753,0.353224,3072,308,526,225
1,xgboost,dev,19264,4131,49,0.5,0.310954,0.585885,0.406279,0.704632,0.378043,2405,975,311,440


**Nhận xét trên dev:** RF có precision khoảng 0,42 nhưng recall khoảng 0,30; XGBoost có recall khoảng 0,59 và precision khoảng 0,31. F1 tương ứng khoảng 0,35 và 0,41. Đây là kết quả tại ngưỡng 0,5, chưa phải kết luận sau tinh chỉnh. Các metric được tính cho lớp dương **Withdrawn**.

## Feature importance
Bảng dưới đây liệt kê 10 feature có importance lớn nhất của mỗi mô hình. Tên cột là tên sau mã hóa; importance mô tả mức mô hình sử dụng feature, không chứng minh quan hệ nhân quả.

In [3]:
importance.groupby('model', group_keys=False).head(10)

,model,feature,importance
36,random_forest,num_raw__date_registration,0.067339
43,random_forest,num_raw__avg_score,0.055542
25,random_forest,num_raw__clicks_per_active_day,0.054068
37,random_forest,num_raw__click_trend,0.054034
23,random_forest,num_raw__total_clicks,0.053034
28,random_forest,num_raw__clicks_w3,0.047480
26,random_forest,num_raw__clicks_w1,0.046912
27,random_forest,num_raw__clicks_w2,0.046267
29,random_forest,num_raw__clicks_w4,0.044607
30,random_forest,num_raw__n_distinct_sites,0.042955


**Đầu ra:** `results/rf_xgb_baseline_dev.csv` và `results/rf_xgb_feature_importance.csv`. Bản baseline 68 feature trước khi dùng preprocessor chung được lưu riêng với hậu tố `legacy_68`.

**Tiếp theo:** Dùng các fold train để chọn cấu hình; sau đó chọn ngưỡng trên dev, thực nghiệm PCA và ablation. Chỉ đánh giá test sau khi khóa lựa chọn. LR dùng nhánh `linear` của preprocessor, còn hai mô hình cây dùng nhánh `tree`, nên cách biến đổi số khác nhau theo loại mô hình.

## Kiểm tra Stratified Group K-fold trên train

Chia 5 fold với `groups=id_student`, `random_state=42`. Mỗi sinh viên chỉ thuộc một validation fold, kể cả khi có nhiều lượt đăng ký. Nhãn được cân bằng ở cấp lượt đăng ký. Bước này chỉ tạo và kiểm tra fold từ train; không huấn luyện mô hình và không dùng dev/test.

In [4]:
from src.models.group_cv import build_folds

fold_summary = build_folds(DATASET, RESULTS)
display(fold_summary)
assert (fold_summary.student_overlap == 0).all()


,fold,train_rows,validation_rows,train_students,validation_students,train_positive,validation_positive,train_positive_rate,validation_positive_rate,student_overlap
0,1,15411,3853,13922,3472,2866,716,0.185971,0.185829,0
1,2,15410,3854,13906,3488,2865,717,0.185918,0.186040,0
2,3,15412,3852,13925,3469,2866,716,0.185959,0.185877,0
3,4,15412,3852,13907,3487,2866,716,0.185959,0.185877,0
4,5,15411,3853,13916,3478,2865,717,0.185906,0.186089,0


**Đầu ra:** `dataset/train_cv_folds.csv` lưu phân fold theo `id_student` và được Git bỏ qua; `results/train_cv_fold_summary.csv` chỉ chứa số liệu tổng hợp, có thể đưa vào báo cáo. Mỗi validation fold có khoảng 3.852–3.854 lượt đăng ký và 3.469–3.488 sinh viên.

## So sánh cấu hình bằng 5 fold train

Thử 3 cấu hình RF và 3 cấu hình XGBoost, mỗi cấu hình trên 5 fold đã lưu. Ở từng fold, preprocessor `tree` được **fit lại chỉ trên phần train của fold**; class weight của RF và `scale_pos_weight` của XGBoost cũng tính từ phần train đó. Cả hai mô hình dùng cùng 33 cột gốc, cho ra 49 feature sau biến đổi trong bộ dữ liệu hiện tại. Chọn cấu hình theo PR-AUC trung bình; F1 tại ngưỡng 0,5 chỉ là chỉ số tham khảo. Chưa dùng dev hoặc test để chọn cấu hình.

In [5]:
from src.models.cv_rf_xgb import run_cv

cv_folds, cv_summary, cv_selected = run_cv(DATASET, RESULTS)
display(cv_summary[["model", "config", "pr_auc_mean", "pr_auc_std", "f1_mean", "recall_mean", "parameters"]].round(4))
display(cv_selected[["model", "config", "pr_auc_mean", "parameters"]].round(4))

,model,config,pr_auc_mean,pr_auc_std,f1_mean,recall_mean,parameters
0,random_forest,rf_leaf5_200,0.3734,0.0224,0.4136,0.4528,"{""max_depth"": null, ""min_samples_leaf"": 5, ""n_..."
1,random_forest,rf_depth12_200,0.3720,0.0166,0.4204,0.4950,"{""max_depth"": 12, ""min_samples_leaf"": 2, ""n_es..."
2,random_forest,rf_baseline_100,0.3598,0.0223,0.3514,0.2984,"{""max_depth"": null, ""min_samples_leaf"": 1, ""n_..."
3,xgboost,xgb_depth4_200,0.3756,0.0145,0.4201,0.5885,"{""learning_rate"": 0.05, ""max_depth"": 4, ""n_est..."
4,xgboost,xgb_baseline_100,0.3752,0.0150,0.4211,0.6067,"{""learning_rate"": 0.1, ""max_depth"": 3, ""n_esti..."
5,xgboost,xgb_depth3_200,0.3740,0.0118,0.4192,0.6039,"{""learning_rate"": 0.05, ""max_depth"": 3, ""n_est..."


,model,config,pr_auc_mean,parameters
0,random_forest,rf_leaf5_200,0.3734,"{""max_depth"": null, ""min_samples_leaf"": 5, ""n_..."
1,xgboost,xgb_depth4_200,0.3756,"{""learning_rate"": 0.05, ""max_depth"": 4, ""n_est..."


**Kết quả:** RF chọn `rf_leaf5_200` (PR-AUC trung bình 0,3734), XGBoost chọn `xgb_depth4_200` (0,3756). XGBoost baseline đạt 0,3752 nên chênh lệch rất nhỏ; chưa thể kết luận cấu hình mới vượt trội ổn định. Các cấu hình này sẽ được kiểm tra tiếp trên dev, nơi chọn ngưỡng. Test vẫn được giữ cho lần đánh giá cuối.

**Đầu ra:** `results/rf_xgb_cv_folds.csv` (30 dòng), `results/rf_xgb_cv_summary.csv` (6 cấu hình) và `results/rf_xgb_cv_selected.csv` (2 cấu hình).

## Bước 4 — PCA, ablation và ngưỡng trên dev

**Ablation có đúng ba trường hợp trên feature gốc:** (1) đầy đủ 33 cột gốc (`full_original`), (2) chỉ nhân khẩu học và `code_module` (`demographic_only`), (3) chỉ hành vi VLE + bài đánh giá và `code_module` (`behavior_only`). Mã hóa làm các trường hợp này lần lượt có 49, 28 và 27 feature. `code_module` là biến bối cảnh chung. So sánh **gốc/PCA là một thí nghiệm riêng**: `pca.joblib` thay 22 cột số đã chuẩn hóa bằng 11 thành phần, giữ nguyên cột phân loại/cờ. Chọn nhánh theo PR-AUC trên dev, rồi chọn ngưỡng theo F1 trên dev.

In [6]:
from src.models.rf_xgb_stage4 import run_dev

dev_options, dev_thresholds, dev_selected, model_lock = run_dev(
    DATASET, RESULTS, RESULTS / "rf_xgb_cv_selected.csv"
)
representation = dev_options.loc[dev_options.experiment == "representation"]
ablation = dev_options.loc[dev_options.experiment == "ablation"]
print("So sánh gốc và PCA")
display(representation[["model", "branch", "n_features", "pr_auc", "f1"]].round(4))
print("Ablation: đúng ba trường hợp cho mỗi mô hình")
display(ablation[["model", "branch", "n_features", "pr_auc", "f1"]].round(4))
print("Cấu hình khóa sau bước chọn ngưỡng")
display(dev_selected[["model", "config", "branch", "threshold", "precision", "recall", "f1", "pr_auc"]].round(4))

So sánh gốc và PCA


,model,branch,n_features,pr_auc,f1
0,random_forest,raw,49,0.3680,0.3966
1,random_forest,pca,38,0.3578,0.3849
5,xgboost,raw,49,0.3736,0.4045
6,xgboost,pca,38,0.3646,0.3934


Ablation: đúng ba trường hợp cho mỗi mô hình


,model,branch,n_features,pr_auc,f1
2,random_forest,full_original,49,0.3680,0.3966
3,random_forest,demographic_only,28,0.2790,0.3378
4,random_forest,behavior_only,27,0.3438,0.3684
7,xgboost,full_original,49,0.3736,0.4045
8,xgboost,demographic_only,28,0.2953,0.3639
9,xgboost,behavior_only,27,0.3552,0.3936


Cấu hình khóa sau bước chọn ngưỡng


,model,config,branch,threshold,precision,recall,f1,pr_auc
0,random_forest,rf_leaf5_200,raw,0.46,0.3444,0.5113,0.4116,0.3680
1,xgboost,xgb_depth4_200,raw,0.49,0.3113,0.5952,0.4088,0.3736


**Kết luận trên dev:** Trong ablation, trường hợp đầy đủ cột gốc có PR-AUC cao nhất cho cả hai mô hình. Ở thí nghiệm riêng về biểu diễn, 49 feature gốc thắng PCA (RF: 0,3680 so với 0,3578; XGBoost: 0,3736 so với 0,3646). Ngưỡng được khóa: RF 0,46; XGBoost 0,49. Các lựa chọn này dùng dev nên kết quả dev có thể lạc quan; đánh giá cuối cần dùng test.

**Đầu ra ablation:** `results/rf_xgb_ablation_dev.csv` có đúng 6 dòng (3 trường hợp × 2 mô hình). `results/rf_xgb_dev_options.csv` lưu thêm thí nghiệm gốc/PCA.

## Bước 5 — kết quả test đã khóa và kiểm tra năm

Cấu hình và ngưỡng trong `results/rf_xgb_model_lock.json` được ghi **trước** kết quả test. Tập test chính đã được đánh giá một lần; notebook chỉ đọc bảng kết quả, không huấn luyện hoặc đánh giá lại test. Kiểm tra 2013→2014 là phân tích phụ: fit trên train 2013, đánh giá test 2014, fit lại preprocessor chỉ trên 2013.

In [7]:
import pandas as pd

test_result = pd.read_csv(RESULTS / "rf_xgb_test_metrics.csv")
temporal_result = pd.read_csv(RESULTS / "rf_xgb_temporal_2013_to_2014.csv")
benchmark_result = pd.read_csv(RESULTS / "rf_xgb_benchmark_summary.csv")
display(test_result[["model", "n_test", "threshold", "precision", "recall", "f1", "roc_auc", "pr_auc", "tn", "fp", "fn", "tp"]].round(4))
display(temporal_result[["model", "n_train", "n_eval", "precision", "recall", "f1", "roc_auc", "pr_auc"]].round(4))
display(benchmark_result[["model", "stage", "runs", "wall_median_s", "cpu_median_percent", "peak_rss_median_mb", "throughput_median_rows_s"]].round(3))

,model,n_test,threshold,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp
0,random_forest,4158,0.46,0.3559,0.5057,0.4178,0.696,0.3705,2643,724,391,400
1,xgboost,4158,0.49,0.3242,0.5992,0.4208,0.706,0.3766,2379,988,317,474


,model,n_train,n_eval,precision,recall,f1,roc_auc,pr_auc
0,random_forest,8332,2380,0.3173,0.5133,0.3922,0.6741,0.3423
1,xgboost,8332,2380,0.3224,0.5440,0.4049,0.6582,0.3494


,model,stage,runs,wall_median_s,cpu_median_percent,peak_rss_median_mb,throughput_median_rows_s
0,random_forest,fit,3,4.065,101.347,273.312,4738.716
1,random_forest,infer,3,1.638,101.335,273.332,50435.459
2,random_forest,read_validate,3,0.083,96.236,274.145,331449.793
3,random_forest,transform,3,0.104,98.289,280.465,225728.207
4,xgboost,fit,3,0.391,101.844,293.098,49222.080
5,xgboost,infer,3,0.122,104.560,293.242,676156.880
6,xgboost,read_validate,3,0.084,102.770,300.492,328871.198
7,xgboost,transform,3,0.086,95.563,294.453,272437.286


**Diễn giải:** XGBoost đạt recall test cao hơn RF (0,5992 so với 0,5057), đồng thời nhiều cảnh báo sai hơn (988 so với 724). F1 của hai mô hình gần nhau (0,4208 và 0,4178). Ở phép kiểm tra 2013→2014, cả hai giảm hiệu năng; `code_module=CCC` chưa thấy trong train 2013 nên cách mã hóa danh mục lạ là một giới hạn. Benchmark báo trung vị ba lượt, suy luận dùng 20 batch/lượt và giới hạn luồng tính toán một lõi. Chi tiết phương pháp, bảng min/max và giới hạn nằm trong `docs/report/rf_xgb_methods_results.md`.